# 04 — Large Problems: SparseIsing, HPC Scaling, and Real Topologies

**What you'll learn:**
- When and why to use `SparseIsing` vs `DenseIsing`
- How to encode real-world graph problems (Max-Cut, Chimera, Pegasus)
- How to scale to thousands/millions of spins using the HPC launcher
- Memory and time complexity analysis
- SLURM deployment walkthrough

In [ ]:
import os, sys
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_PY   = os.path.join(_ROOT, 'python')
if _PY not in sys.path:
    sys.path.insert(0, _PY)

import numpy as np
import time
import matplotlib.pyplot as plt
from qanneal import DenseIsing, SparseIsing, SparseEdge, solve, auto_schedule_sqa_tuned

---

## 1 — DenseIsing vs SparseIsing: Memory and Speed

| Representation | Memory | `delta_energy()` | Best for |
|---------------|--------|-----------------|----------|
| `DenseIsing` | O(n²) | O(n) | Fully connected, n ≤ 3 000 |
| `SparseIsing` | O(n + \|E\|) | O(degree) | Sparse graphs, n ≥ 1 000 |

For a 1000-spin fully connected graph: DenseIsing needs 8 MB; SparseIsing with degree-10 needs ~160 KB.

In [ ]:
# Memory comparison
import sys as _sys

for n in [100, 500, 1000, 2000]:
    dense_mb = n * n * 8 / 1e6   # 8 bytes per float64
    # Sparse: h array + edges (n*degree triples)
    degree = 10
    sparse_mb = (n * 8 + n * degree * 3 * 8) / 1e6
    ratio = dense_mb / sparse_mb
    print(f'n={n:5d}: Dense={dense_mb:6.1f} MB  Sparse={sparse_mb:6.2f} MB  ratio={ratio:.0f}x')

In [ ]:
# Speed comparison: construction and energy evaluation
def make_dense_random(n, seed=0):
    rng = np.random.default_rng(seed)
    h = rng.uniform(-1, 1, n)
    J = rng.uniform(-1, 1, (n, n))
    J = (J + J.T) / 2   # symmetrize
    return DenseIsing(h, J)

def make_sparse_kregular(n, k=10, seed=0):
    rng = np.random.default_rng(seed)
    h = rng.uniform(-1, 1, n)
    edge_dict = {}
    for i in range(n):
        for _ in range(k):
            j = int((i + int(rng.integers(1, n))) % n)
            key = (min(i, j), max(i, j))
            if key[0] != key[1]:
                edge_dict[key] = float(rng.uniform(-1, 1))
    edges = [SparseEdge(i, j, v) for (i, j), v in edge_dict.items()]
    return SparseIsing(h, edges, n)

print('Testing n=500...')
n_test = 500

t0 = time.perf_counter()
dense = make_dense_random(n_test)
t_dense_build = time.perf_counter() - t0

t0 = time.perf_counter()
sparse = make_sparse_kregular(n_test, k=10)
t_sparse_build = time.perf_counter() - t0

# Test energy evaluation speed
spins = [1 if i % 2 == 0 else -1 for i in range(n_test)]

t0 = time.perf_counter()
for _ in range(100):
    dense.energy(spins)
t_dense_eval = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(100):
    sparse.energy(spins)
t_sparse_eval = (time.perf_counter() - t0) / 100 * 1000

print(f'Build time: Dense={t_dense_build*1000:.1f}ms  Sparse={t_sparse_build*1000:.1f}ms')
print(f'energy() time: Dense={t_dense_eval:.3f}ms  Sparse={t_sparse_eval:.3f}ms  (speedup: {t_dense_eval/t_sparse_eval:.1f}x)')

---

## 2 — Real Topology: D-Wave Chimera

The **Chimera** graph is the native topology of D-Wave 2000Q.
A Chimera C_{m,m,4} graph has 8m² qubits arranged in m×m unit cells,
each cell containing a K_{4,4} bipartite graph.

```
Unit cell:
  h₀ ──┬── h₁ ──┬── h₂ ──┬── h₃  (horizontal qubits)
   ×   │    ×   │    ×   │    ×
  v₀   v₁   v₂   v₃               (vertical qubits)
```

Each horizontal qubit connects to all 4 vertical qubits in its cell (16 internal edges).
Horizontal qubits also connect to the same qubit in horizontally adjacent cells.

In [ ]:
def make_chimera(m, seed=0, scale=1.0):
    """Chimera C_{m,m,4} with random ±1 couplings."""
    rng = np.random.default_rng(seed)
    n = 8 * m * m
    h = rng.uniform(-scale, scale, n)

    def qubit(row, col, side, k):
        return (row * m + col) * 8 + side * 4 + k

    edge_set = {}
    for row in range(m):
        for col in range(m):
            # Intra-cell: bipartite K_{4,4}
            for k0 in range(4):
                for k1 in range(4):
                    q0 = qubit(row, col, 0, k0)
                    q1 = qubit(row, col, 1, k1)
                    key = (min(q0,q1), max(q0,q1))
                    edge_set[key] = float(rng.uniform(-scale, scale))
            # Horizontal inter-cell
            if col + 1 < m:
                for k in range(4):
                    q0 = qubit(row, col, 0, k)
                    q1 = qubit(row, col+1, 0, k)
                    key = (min(q0,q1), max(q0,q1))
                    edge_set[key] = float(rng.uniform(-scale, scale))
            # Vertical inter-cell
            if row + 1 < m:
                for k in range(4):
                    q0 = qubit(row, col, 1, k)
                    q1 = qubit(row+1, col, 1, k)
                    key = (min(q0,q1), max(q0,q1))
                    edge_set[key] = float(rng.uniform(-scale, scale))

    edges = [SparseEdge(i, j, v) for (i, j), v in edge_set.items()]
    return SparseIsing(h, edges, n)

# Chimera C_{4,4,4}: 128 qubits
m = 4
chimera_ising = make_chimera(m, seed=42)
print(f'Chimera C_{{{m},{m},4}}: n={chimera_ising.size()} qubits')

# Solve with SQAPT
schedule = auto_schedule_sqa_tuned(chimera_ising, mode='balanced')
result = solve(
    chimera_ising, method='sqapt',
    reads=20, sweeps_per_beta=60, worldline_sweeps=5,
    trotter_slices=24, replicas=8, pt_steps=60,
    seed=0, progress=False,
)
print(f'Best energy: {result.best_energy:.4f}')
print(f'Energy distribution: mean={np.mean(result.energies):.3f}  std={np.std(result.energies):.3f}')

---

## 3 — Max-Cut Problem

**Max-Cut**: Partition graph nodes into two sets to maximise the number of edges between sets.

**Ising encoding**: Set $J_{ij} = -1$ for all edges. Minimising Ising energy = maximising cut.

Reason: $E = \sum_{ij} J_{ij} s_i s_j = -\sum_{(i,j) \in E} s_i s_j$.
When $s_i \neq s_j$ (edge is cut), $s_i s_j = -1$, contributing $+1$ to $-E$.

In [ ]:
def make_maxcut_ising(n, degree, seed=0):
    """Max-Cut as sparse Ising: J_ij = -1 for all edges."""
    rng = np.random.default_rng(seed)
    h = np.zeros(n)
    edge_dict = {}
    for i in range(n):
        for _ in range(degree * 2):
            j = int((i + int(rng.integers(1, n))) % n)
            key = (min(i, j), max(i, j))
            if key[0] != key[1]:
                edge_dict[key] = -1.0
    edges = [SparseEdge(i, j, v) for (i, j), v in edge_dict.items()]
    return SparseIsing(h, edges, n), edge_dict

n_mc = 100
mc_ising, edge_dict = make_maxcut_ising(n_mc, degree=5, seed=7)
n_edges = len(edge_dict)
print(f'Max-Cut: n={n_mc} nodes, {n_edges} edges, avg_degree={2*n_edges/n_mc:.1f}')

# Solve
r_mc = solve(
    mc_ising, method='sqapt',
    reads=30, sweeps_per_beta=60, worldline_sweeps=5,
    trotter_slices=16, replicas=8, pt_steps=50,
    seed=0, progress=False,
)

spins = r_mc.best_sample
cut_size = sum(1 for (i, j) in edge_dict if spins[i] != spins[j])
print(f'Max-Cut found: {cut_size}/{n_edges} edges ({100*cut_size/n_edges:.1f}%)')
print(f'Ising energy: {r_mc.best_energy:.3f}  (= −cut_size = −{cut_size})')

---

## 4 — Scaling Study: Wall Time vs Problem Size

Understanding how wall time scales with n helps plan HPC resources.

**SQA complexity per sweep**:
- Dense Ising: O(n) per spin flip × n spins × slices × reads = O(n² × M)
- Sparse Ising: O(degree) per spin flip × n spins × slices × reads = O(n × degree × M)

In [ ]:
sizes = [50, 100, 200, 400, 800, 1500]
READS = 4
SWEEPS = 30

print(f'Scaling: SQA, {READS} reads, {SWEEPS} sweeps/β, M=16')
print(f'{"n":>6}  {"edges":>8}  {"time_s":>8}  {"ns/step":>10}')
print('-' * 40)

times = []
ns_list = []
for n in sizes:
    ising_n, _ = make_maxcut_ising(n, degree=6, seed=0)
    n_e = ising_n.size()  # same as n
    sch = auto_schedule_sqa_tuned(ising_n, mode='fast')

    t0 = time.perf_counter()
    solve(ising_n, method='sqa', reads=READS, sweeps_per_beta=SWEEPS,
          trotter_slices=16, schedule=sch, seed=0, progress=False)
    dt = time.perf_counter() - t0

    n_steps = len(sch.betas)
    ns_per_step = dt / (n_steps * READS) * 1e9
    times.append(dt)
    ns_list.append(n)
    print(f'{n:>6}  {len(edge_dict):>8}  {dt:>8.3f}s  {ns_per_step:>10.0f}ns')

In [ ]:
# Plot scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.loglog(ns_list, times, 'o-', color='#4c8bf5', lw=2, ms=7, label='SQA (sparse)')
# O(n) and O(n log n) reference lines
t0_ref = times[0] / ns_list[0]
n_arr = np.array(ns_list, dtype=float)
ax.loglog(ns_list, t0_ref * n_arr, '--', color='#aaa', lw=1.5, label='O(n)')
ax.loglog(ns_list, t0_ref * n_arr * np.log2(np.maximum(n_arr, 2)), ':', color='#888', lw=1.5, label='O(n log n)')
ax.set_xlabel('n (spins)')
ax.set_ylabel('Wall time (s)')
ax.set_title('SQA scaling with SparseIsing')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Extrapolation
ax2 = axes[1]
ns_extrap = np.array([100, 500, 1000, 5000, 10000, 50000, 100000])
# Fit linear extrapolation on log-log scale
slope = np.polyfit(np.log(ns_list), np.log(times), 1)[0]
log_intercept = np.polyfit(np.log(ns_list), np.log(times), 1)[1]
t_extrap = np.exp(log_intercept) * ns_extrap ** slope
ax2.loglog(ns_extrap, t_extrap, 'o--', color='#c84b31', lw=2, ms=5, label=f'Extrapolated (n^{slope:.2f})')
ax2.loglog(ns_list, times, 's', color='#4c8bf5', ms=8, label='Measured')
ax2.set_xlabel('n (spins)')
ax2.set_ylabel('Estimated wall time (s)')
ax2.set_title('Extrapolated scaling')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

# Print extrapolation table
for n_ext, t_ext in zip(ns_extrap, t_extrap):
    unit = 's'
    val = t_ext
    if val > 3600: val /= 3600; unit = 'h'
    elif val > 60: val /= 60; unit = 'min'
    print(f'n={n_ext:>8,}: ~{val:.1f} {unit} per read')

fig.tight_layout()
plt.show()

---

## 5 — HPC Deployment: The hpc_sqa_launcher

For n > 5 000, use the built-in HPC launcher. It handles:
- `SparseIsing` problem generation
- Multiprocessing (single node) or MPI (multi-node)
- Pickling-safe serialisation of the problem
- JSON + numpy output for post-processing

In [ ]:
# Single-node example with the HPC launcher
import subprocess, json

launcher = os.path.join(_ROOT, 'examples', 'python', 'hpc_sqa_launcher.py')

cmd = [
    sys.executable, launcher,
    '--n', '1000',
    '--degree', '8',
    '--method', 'sqa',
    '--reads', '4',
    '--mode', 'fast',
    '--workers', '1',
    '--out', '/tmp/nb04_hpc_run',
]

print('Running:', ' '.join(cmd[2:]))  # skip python path
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:', proc.stderr[:500])

In [ ]:
# Load and inspect the results
with open('/tmp/nb04_hpc_run.json') as f:
    result_json = json.load(f)

print('Results from HPC launcher:')
for k, v in result_json.items():
    if k != 'all_energies':
        print(f'  {k:18s}: {v}')

# Load spins
spins = np.load('/tmp/nb04_hpc_run_spins.npy')
print(f'\nSpin vector: shape={spins.shape}  dtype={spins.dtype}')
print(f'  +1 spins: {np.sum(spins==1)}  |  -1 spins: {np.sum(spins==-1)}')

In [ ]:
# Plot energy distribution
energies = result_json['all_energies']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(energies, bins=max(5, len(energies)//2), color='#4c8bf5', edgecolor='white')
ax.axvline(result_json['best_energy'], color='#c84b31', lw=2, linestyle='--',
           label=f'Best: {result_json["best_energy"]:.2f}')
ax.axvline(result_json['energy_mean'], color='#2ca02c', lw=2, linestyle=':',
           label=f'Mean: {result_json["energy_mean"]:.2f}')
ax.set_xlabel('Energy')
ax.set_ylabel('Count')
ax.set_title(f'Energy distribution: n={result_json["n_spins"]} spins, {result_json["reads"]} reads')
ax.legend()
fig.tight_layout()
plt.show()

---

## 6 — SLURM Deployment Guide

### Option A: MPI (requires mpi4py)

```bash
# Edit scripts/slurm/run_sqa_hpc.sh with your cluster's modules
# Then submit:
sbatch scripts/slurm/run_sqa_hpc.sh

# Or override parameters inline:
sbatch --export=N_SPINS=50000,METHOD=sqapt,READS=16,N=4 \
       scripts/slurm/run_sqa_hpc.sh
```

This runs 4 nodes × 8 ranks = 32 MPI processes, each doing 16 reads = **512 total reads** in parallel.

### Option B: Job array (no mpi4py needed — most portable)

```bash
sbatch scripts/slurm/run_sqa_array.sh
# Creates: results/run_0.json ... results/run_31.json

# After all tasks complete (watch with: squeue -u $USER):
python examples/python/merge_array_results.py results/run_*.json --out merged_best
```

### Option C: Large n with save/load

```bash
# Generate and save the problem once
python examples/python/hpc_sqa_launcher.py \
  --n 100000 --degree 10 --save-problem --out results/my_problem

# Then use from_file across many SLURM tasks
export PROBLEM_FILE=results/my_problem_problem.npz
export PROBLEM_TYPE=from_file
sbatch scripts/slurm/run_sqa_array.sh
```

### Resource planning table

| n spins | degree | Method | Reads | Time/rank | Nodes × tasks |
|---------|--------|--------|-------|-----------|---------------|
| 1 000 | 10 | SQA fast | 32 | ~0.3 min | 1 × 4 |
| 5 000 | 10 | SQA balanced | 32 | ~5 min | 1 × 8 |
| 10 000 | 10 | SQA balanced | 16 | ~20 min | 2 × 8 |
| 50 000 | 8 | SQA fast | 8 | ~60 min | 4 × 16 |
| 100 000 | 6 | SQA fast | 4 | ~3 hr | 8 × 16 |

---

## 7 — Scaling Benchmark Results

The following data was captured by running:
```bash
python examples/python/hpc_sqa_launcher.py --scaling --method sqa --mode fast --scaling-reads 4
```

In [ ]:
# Load the pre-computed scaling results
scaling_file = os.path.join(_ROOT, 'examples', 'python', 'hpc_scaling_sqa_fast.json')

if os.path.exists(scaling_file):
    with open(scaling_file) as f:
        scaling_data = json.load(f)

    ns    = [d['n']           for d in scaling_data]
    times = [d['wall_time_s'] for d in scaling_data]
    Es    = [d['best_energy'] for d in scaling_data]

    print('n        time_s')
    for n, t in zip(ns, times):
        print(f'{n:>8,}  {t:>8.2f}s')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.loglog(ns, times, 'o-', color='#4c8bf5', lw=2, ms=7)
    n_arr = np.array(ns, dtype=float)
    t0_ref = times[0] / ns[0]
    ax.loglog(ns, t0_ref * n_arr, '--', color='#aaa', lw=1.5, label='O(n)')
    ax.loglog(ns, t0_ref * n_arr * np.log2(np.maximum(n_arr, 2)), ':', color='#888', lw=1.5, label='O(n log n)')
    ax.set_xlabel('n (spins)')
    ax.set_ylabel('Wall time (s)')
    ax.set_title('SQA [fast] scaling — measured data')
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)

    ax2 = axes[1]
    # Normalise energy by n to see energy/spin
    energy_per_spin = np.array(Es) / np.array(ns, dtype=float)
    ax2.semilogx(ns, energy_per_spin, 's-', color='#c84b31', lw=2, ms=7)
    ax2.set_xlabel('n (spins)')
    ax2.set_ylabel('Energy per spin')
    ax2.set_title('Solution quality vs problem size')
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print(f'Scaling file not found at: {scaling_file}')
    print('Run: python examples/python/hpc_sqa_launcher.py --scaling --method sqa --mode fast')

---

## 8 — Summary

| Problem size | Recommendation |
|-------------|----------------|
| n ≤ 200 | `DenseIsing`, any method, brute-force for oracle |
| 200 < n ≤ 1 000 | `DenseIsing` or `SparseIsing`, `sqapt`, `mode="balanced"` |
| 1 000 < n ≤ 10 000 | `SparseIsing`, `sqa` or `sqapt`, multiprocessing workers |
| n > 10 000 | `SparseIsing`, `sqa fast`, HPC launcher + SLURM |
| n > 100 000 | `SparseIsing`, sparse graph only (degree ≤ 15), multi-node MPI |

**Key principle**: total reads = `mpi_size × workers × reads_per_worker`.
More reads always improves solution quality — parallelize freely.